In [2]:
import argparse
import torch
import time
import torchvision.models as models

In [5]:
def get_model(name: str) -> torch.nn.Module:
    name = name.lower()
    if name == "resnet18":
        return models.resnet18
    elif name == "resnet50":
        return models.resnet50
    raise ValueError(f"Unknown model: {name}")

@torch.no_grad()
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, default="resnet18")
    parser.add_argument("--batch", type=int, default=1)
    parser.add_argument("--resolution", type=int, default=224)
    parser.add_argument("--iters", type=int, default=200)
    parser.add_argument("--warmup", type=int, default=30)
    args = parser.parse_args()



In [24]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import requests
from io import BytesIO

# 1) Load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.eval()

# 2) Download an example image (you can replace this with your own file)

url = "https://upload.wikimedia.org/wikipedia/commons/a/a0/SStringray.jpg"
headers = {"User-Agent": "Mozilla/5.0"}

with requests.get(url, headers=headers, stream=True, timeout=30) as r:
    r.raise_for_status()
    r.raw.decode_content = True
    img = Image.open(r.raw).convert("RGB")
    
# 3) Preprocessing: resize, center crop, normalize (ImageNet stats)
preprocess = weights.transforms()
input_tensor = preprocess(img).unsqueeze(0)  # add batch dimension

# 4) Run inference
with torch.no_grad():
    output = model(input_tensor)

# 5) Get predicted class
probs = torch.softmax(output, dim=1)
top_prob, top_idx = torch.max(probs, dim=1)

# 6) Load ImageNet class names
categories = weights.meta["categories"]
print(categories)
predicted_class = categories[top_idx.item()]

print(f"Predicted class: {predicted_class}")
print(f"Confidence: {top_prob.item() * 100:.2f}%")

['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead', 'electric ray', 'stingray', 'cock', 'hen', 'ostrich', 'brambling', 'goldfinch', 'house finch', 'junco', 'indigo bunting', 'robin', 'bulbul', 'jay', 'magpie', 'chickadee', 'water ouzel', 'kite', 'bald eagle', 'vulture', 'great grey owl', 'European fire salamander', 'common newt', 'eft', 'spotted salamander', 'axolotl', 'bullfrog', 'tree frog', 'tailed frog', 'loggerhead', 'leatherback turtle', 'mud turtle', 'terrapin', 'box turtle', 'banded gecko', 'common iguana', 'American chameleon', 'whiptail', 'agama', 'frilled lizard', 'alligator lizard', 'Gila monster', 'green lizard', 'African chameleon', 'Komodo dragon', 'African crocodile', 'American alligator', 'triceratops', 'thunder snake', 'ringneck snake', 'hognose snake', 'green snake', 'king snake', 'garter snake', 'water snake', 'vine snake', 'night snake', 'boa constrictor', 'rock python', 'Indian cobra', 'green mamba', 'sea snake', 'horned viper', 'diamondback', 

In [27]:
import torch
import torchvision.models as models
from fvcore.nn import FlopCountAnalysis

model = models.resnet18(weights=None).eval()
x = torch.randn(1, 3, 224, 224)

flop_an = FlopCountAnalysis(model, x)

# Explicitly set unsupported ops to 0 FLOPs
flop_an.set_op_handle("aten::max_pool2d", lambda *args, **kwargs: 0)
flop_an.set_op_handle("aten::add_", lambda *args, **kwargs: 0)

flops = flop_an.total()
print("GFLOPs:", flops / 1e9)

GFLOPs: 1.819065856
